# Import libraries and dataset into environment

In [1]:
import dill
import os
import sys
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.linear_model import LogisticRegression
import math as mt
import shap
from sklearn.metrics import confusion_matrix, roc_curve, auc
from MLstatkit import Bootstrapping
import defined_functions
from defined_functions import sum_metric, met_collate_func
defined_functions.pd = pd
import gc
from joblib import Parallel, delayed, externals
import multiprocessing
num_cores = multiprocessing.cpu_count() - 1
import defined_functions
from defined_functions import sum_metric, met_collate_func
defined_functions.pd = pd

In [2]:
path = os.getcwd()
sys.path.append(path)

save_file = os.path.join(path, "session.pkl")

with open(save_file, "rb") as f:
    state = dill.load(f)

split_list_valid_smote = state["split_list_valid_smote"]
split_list_valid_smote_final = state["split_list_valid_smote_final"]
split_list_fil_valid_smote_final = state["split_list_fil_valid_smote_final"]
split_list_sim_onset_valid_smote_final = state["split_list_sim_onset_valid_smote_final"]
split_list_vldiag_valid_smote_final = state["split_list_vldiag_valid_smote_final"]
split_list_diag1_valid_smote_final = state["split_list_diag1_valid_smote_final"]
split_list_diag2_valid_smote_final = state["split_list_diag2_valid_smote_final"]

# Define functions

In [3]:
# Define function to train Ridge regression model
def run_single_ridge_split(i, split):
    train_df = split['train'].copy()
    valid_df = split['valid'].copy()
    test_df = split['test'].copy()

    train_df['outcome'] = train_df['outcome'].cat.reorder_categories(["Non severe", "Severe"], ordered = False)
    valid_df['outcome'] = valid_df['outcome'].cat.reorder_categories(["Non severe", "Severe"], ordered = False)
    test_df['outcome'] = test_df['outcome'].cat.reorder_categories(["Non severe", "Severe"], ordered = False)

    train_x = train_df.drop(columns = ['outcome'])
    valid_x = valid_df.drop(columns = ['outcome'])
    test_x = test_df.drop(columns = ['outcome'])

    features = list(train_x.columns)
    cat_cols = ['age', 'gender', 'vaccination', 'comorbidity']
    for col in cat_cols:
        train_x[col] = train_x[col].astype('category')
        valid_x[col] = valid_x[col].astype('category')
        test_x[col] = test_x[col].astype('category')

    train_x = pd.get_dummies(train_x, columns = cat_cols, drop_first = False)
    valid_x = pd.get_dummies(valid_x, columns = cat_cols, drop_first = False)
    test_x = pd.get_dummies(test_x, columns = cat_cols, drop_first = False)

    feature_names = train_x.columns.tolist()

    # Force valid/test to have same columns and same order as train
    valid_x = valid_x.reindex(columns = feature_names, fill_value = 0)
    test_x = test_x.reindex(columns = feature_names, fill_value = 0)
    
    mapping = {"Non severe": 0, "Severe": 1}
    
    train_y = train_df['outcome'].map(mapping).astype(int)
    valid_y = valid_df['outcome'].map(mapping).astype(int)
    test_y = test_df['outcome'].map(mapping).astype(int)
    ridge = LogisticRegression(
        penalty = 'l2',  # Perform L1-regularization/Lasso regression
        solver = 'lbfgs',
        l1_ratio = 0.0, # Elastic-Net mixing parameter
        C = 0.1,   # Inverse of regularization strength
        class_weight = {0: 1, 1: 1.25},    # Define wights associated with classes
        max_iter = 1000,
        random_state = 123,
        verbose = 0
    )

    ridge_model = ridge.fit(train_x, train_y)

    # Make predictions on Testing set
    ridge_pred_prob = ridge_model.predict_proba(test_x)[:, 1]

    # Calculate Area Under the ROC curve
    fpr, tpr_curve, _ = roc_curve(test_y, ridge_pred_prob, pos_label = 1)
    auc_val_ridge = auc(fpr, tpr_curve)

    # Store standard structure dictionary
    roc_container = {
        'actual': test_y,
        'probabilities': ridge_pred_prob
    }
    
    # Calculate Area Under the Precision-Recall Curve (AUPRC / PR-AUC)
    auprc_val, prc_ci_lower, prc_ci_upper = Bootstrapping(test_y, ridge_pred_prob, 'pr_auc')

    # Convert predicted probabilities to the labels
    ridge_pred = ridge_model.predict(test_x).astype(int)
    ridge_pred_res = np.where(ridge_pred == 0, "Non severe", "Severe")
    
    # Compute Confusion Matrix (Test)
    tn, fp, fn, tp = confusion_matrix(test_y, ridge_pred, labels = [0, 1]).ravel()
    
    # Format a formal R-styled evaluation matrix lookup dataframe 
    cfm_ridge = pd.DataFrame(
        [[tn, fp], [fn, tp]], 
        index = ["Non severe", "Severe"], 
        columns = ["Non severe", "Severe"]
    )
    cfm_ridge.index.name = 'Prediction'
    cfm_ridge.columns.name = 'Observed'
    
    # Calculate performance metrics
    accuracy_ridge = (tp + tn) / (tn + fp + fn + tp) if (tn + fp + fn + tp) > 0 else 0
    sensitivity_ridge = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity_ridge = tn / (tn + fp) if (tn + fp) > 0 else 0
    npv_ridge = tn / (tn + fn) if (tn + fn) > 0 else 0
    precision_ridge = tp / (tp + fp) if (tp + fp) > 0 else 0
        
    # Make predictions on Training Set to gather accuracy
    ridge_pred_train_prob = ridge_model.predict_proba(train_x)[:, 1]
    ridge_pred_train = ridge_model.predict(train_x).astype(int)
    tn_tr, fp_tr, fn_tr, tp_tr = confusion_matrix(train_y, ridge_pred_train, labels = [0, 1]).ravel()
    accuracy_ridge_train = (tp_tr + tn_tr) / (tn_tr + fp_tr + fn_tr + tp_tr)
    
    masker = shap.maskers.Independent(train_x)
    explainer = shap.LinearExplainer(model = ridge_model, masker = masker)
    shap_values = explainer(train_x).values

    return {
        "model": ridge_model,
        "confusion_matrix": cfm_ridge,
        "accuracy_training": accuracy_ridge_train,
        "accuracy_testing": accuracy_ridge,
        "sensitivity": sensitivity_ridge,
        "specificity": specificity_ridge,
        "precision": precision_ridge,
        "npv": npv_ridge,
        "roc": roc_container,
        "AUC_value": auc_val_ridge,
        "PRC_val": auprc_val,
        "PRC_lower_ci": prc_ci_lower,
        "PRC_upper_ci": prc_ci_upper,
        "SHAP_values": shap_values,
        "prediction": ridge_pred_res,
        "pred_prob": ridge_pred_prob
    }

# Define function to train Ridge regression for 100 times in parallel
def model_func_ridge_tune(data_list):
    
    # Count system resource availability profiles
    num_cores = multiprocessing.cpu_count() - 1
    
    if __name__ == '__main__':
        try:
            result_list = Parallel(n_jobs = num_cores)(
                delayed(run_single_ridge_split)(i, data_list[i]) 
                for i in range(100)
                )
        finally:
            externals.loky.get_reusable_executor().shutdown(wait = True)
            gc.collect()

    return result_list

# Fitting dataset into the model

## Fitting data list without VL information

In [4]:
ridge_fil_list = model_func_ridge_tune(split_list_fil_valid_smote_final)
ridge_fil_met_summary = sum_metric(ridge_fil_list)
ridge_fil_metrics_summary = ridge_fil_met_summary["metric_summary"]
ridge_fil_summary = met_collate_func(ridge_fil_metrics_summary).assign(
    models = "Ridge regression (No VL info & SMOTE)"
)
ridge_fil_summary

/home/blaw004/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
Bootstrapping pr_auc:   0%|          | 0/1000 [00:00<?, ?it/s]/home/blaw004/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=N

bound,Metrics,estimate,lower,upper,models
0,AUPRC_value,36.7286,17.4064,58.7036,Ridge regression (No VL info & SMOTE)
1,AUROC_value,92.1612,82.4295,97.5012,Ridge regression (No VL info & SMOTE)
2,Accuracy,86.8912,80.3474,92.1601,Ridge regression (No VL info & SMOTE)
3,Accuracy_train,87.8702,83.2386,91.8510,Ridge regression (No VL info & SMOTE)
4,NPV,99.4612,98.6607,100.0000,Ridge regression (No VL info & SMOTE)
5,Precision,17.4454,11.5661,25.7143,Ridge regression (No VL info & SMOTE)
6,Sensitivity,84.7000,60.0000,100.0000,Ridge regression (No VL info & SMOTE)
7,Specificity,86.9595,79.7352,92.6869,Ridge regression (No VL info & SMOTE)


## Fitting data list with simulated VL at symptom onset

In [5]:
ridge_vlsymp_sim_list = model_func_ridge_tune(split_list_sim_onset_valid_smote_final)
ridge_vlsymp_sim_met_summary = sum_metric(ridge_vlsymp_sim_list)
ridge_vlsymp_sim_metrics_summary = ridge_vlsymp_sim_met_summary["metric_summary"]
ridge_vlsymp_sim_summary = met_collate_func(ridge_vlsymp_sim_metrics_summary).assign(
    models = "Ridge regression (VL symp simulated & SMOTE)"
)
ridge_vlsymp_sim_summary

/home/blaw004/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/blaw004/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
Bootstrapping pr_auc:  15%|█▌        | 1

bound,Metrics,estimate,lower,upper,models
0,AUPRC_value,37.4123,17.5540,60.8927,Ridge regression (VL symp simulated & SMOTE)
1,AUROC_value,92.0629,82.7718,97.7072,Ridge regression (VL symp simulated & SMOTE)
2,Accuracy,86.9154,80.3474,92.1601,Ridge regression (VL symp simulated & SMOTE)
3,Accuracy_train,87.9600,83.2217,91.7718,Ridge regression (VL symp simulated & SMOTE)
4,NPV,99.4416,98.6348,100.0000,Ridge regression (VL symp simulated & SMOTE)
5,Precision,17.3145,11.7708,24.3200,Ridge regression (VL symp simulated & SMOTE)
6,Sensitivity,84.1000,60.0000,100.0000,Ridge regression (VL symp simulated & SMOTE)
7,Specificity,87.0031,80.2103,93.0140,Ridge regression (VL symp simulated & SMOTE)


## Fitting data list with VL at diagnosis

In [6]:
ridge_vldiag_list = model_func_ridge_tune(split_list_vldiag_valid_smote_final)
ridge_vldiag_met_summary = sum_metric(ridge_vldiag_list)
ridge_vldiag_metrics_summary = ridge_vldiag_met_summary["metric_summary"]
ridge_vldiag_summary = met_collate_func(ridge_vldiag_metrics_summary).assign(
    models = "Ridge regression (VL diag & SMOTE)"
)
ridge_vldiag_summary

/home/blaw004/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
Bootstrapping pr_auc:   0%|          | 0/1000 [00:00<?, ?it/s]/home/blaw004/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=N

bound,Metrics,estimate,lower,upper,models
0,AUPRC_value,41.3730,18.7843,63.1072,Ridge regression (VL diag & SMOTE)
1,AUROC_value,92.8991,83.6643,98.0093,Ridge regression (VL diag & SMOTE)
2,Accuracy,87.6042,81.1103,92.1450,Ridge regression (VL diag & SMOTE)
3,Accuracy_train,88.8603,85.0714,92.2144,Ridge regression (VL diag & SMOTE)
4,NPV,99.4302,98.5988,100.0000,Ridge regression (VL diag & SMOTE)
5,Precision,18.0843,12.3211,25.3365,Ridge regression (VL diag & SMOTE)
6,Sensitivity,83.7000,60.0000,100.0000,Ridge regression (VL diag & SMOTE)
7,Specificity,87.7259,80.8333,92.6869,Ridge regression (VL diag & SMOTE)


## Fitting data list with VL at diagnosis & VL at 1-day after diagnosis

In [7]:
ridge_add1_list = model_func_ridge_tune(split_list_diag1_valid_smote_final)
ridge_add1_met_summary = sum_metric(ridge_add1_list)
ridge_add1_metrics_summary = ridge_add1_met_summary["metric_summary"]
ridge_add1_summary = met_collate_func(ridge_add1_metrics_summary).assign(
    models = "Ridge regression (VL diag + 1 & SMOTE)"
)
ridge_add1_summary

/home/blaw004/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/blaw004/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/blaw004/anaconda3/lib/python3.12/s

bound,Metrics,estimate,lower,upper,models
0,AUPRC_value,41.6144,19.1506,62.4719,Ridge regression (VL diag + 1 & SMOTE)
1,AUROC_value,93.0857,83.5171,98.1519,Ridge regression (VL diag + 1 & SMOTE)
2,Accuracy,87.8671,81.1103,92.4622,Ridge regression (VL diag + 1 & SMOTE)
3,Accuracy_train,89.4585,86.1072,92.8647,Ridge regression (VL diag + 1 & SMOTE)
4,NPV,99.4454,98.6620,100.0000,Ridge regression (VL diag + 1 & SMOTE)
5,Precision,18.4942,12.5836,26.9143,Ridge regression (VL diag + 1 & SMOTE)
6,Sensitivity,84.1000,60.0000,100.0000,Ridge regression (VL diag + 1 & SMOTE)
7,Specificity,87.9844,80.9969,92.8349,Ridge regression (VL diag + 1 & SMOTE)


## Fitting data list with VL at diagnosis & VL at 2-days after diagnosis

In [8]:
ridge_add2_list = model_func_ridge_tune(split_list_diag2_valid_smote_final)
ridge_add2_met_summary = sum_metric(ridge_add2_list)
ridge_add2_metrics_summary = ridge_add2_met_summary["metric_summary"]
ridge_add2_summary = met_collate_func(ridge_add2_metrics_summary).assign(
    models = "Ridge regression (VL diag + 2 & SMOTE)"
)
ridge_add2_summary

/home/blaw004/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
Bootstrapping pr_auc:   0%|          | 0/1000 [00:00<?, ?it/s]/home/blaw004/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=N

bound,Metrics,estimate,lower,upper,models
0,AUPRC_value,42.9488,18.8227,65.1358,Ridge regression (VL diag + 2 & SMOTE)
1,AUROC_value,93.2667,83.9634,98.1994,Ridge regression (VL diag + 2 & SMOTE)
2,Accuracy,88.0967,81.8731,92.4471,Ridge regression (VL diag + 2 & SMOTE)
3,Accuracy_train,89.7638,86.4213,93.0971,Ridge regression (VL diag + 2 & SMOTE)
4,NPV,99.4608,98.6620,100.0000,Ridge regression (VL diag + 2 & SMOTE)
5,Precision,18.8590,12.8882,26.8917,Ridge regression (VL diag + 2 & SMOTE)
6,Sensitivity,84.5000,60.0000,100.0000,Ridge regression (VL diag + 2 & SMOTE)
7,Specificity,88.2087,81.7679,92.8505,Ridge regression (VL diag + 2 & SMOTE)


# Save model trained

In [9]:
path = os.getcwd()

state = {
    "ridge_fil_list": ridge_fil_list,
    "ridge_vlsymp_sim_list": ridge_vlsymp_sim_list,
    "ridge_vldiag_list": ridge_vldiag_list,
    "ridge_add1_list": ridge_add1_list,
    "ridge_add2_list": ridge_add2_list
}

save_file = os.path.join(path, "ridge_trained.pkl")

with open(save_file, "wb") as f:
    dill.dump(state, f)

#print(f"Saved to: {save_file}")

In [10]:
path = os.getcwd()

state = {
    "ridge_fil_summary": ridge_fil_summary,
    "ridge_vlsymp_sim_summary": ridge_vlsymp_sim_summary,
    "ridge_vldiag_summary": ridge_vldiag_summary,
    "ridge_add1_summary": ridge_add1_summary,
    "ridge_add2_summary": ridge_add2_summary
}

save_file = os.path.join(path, "ridge_metric_summary.pkl")

with open(save_file, "wb") as f:
    dill.dump(state, f)

#print(f"Saved to: {save_file}")